# Product Performance ETL

## Purpose
Analyze product-level performance metrics for business analytics, product management, and inventory optimization.

## Input → Output
* **Source:** `big_data.silver.order_products` 
* **Source:** `big_data.silver.products_enriched` 
* **Target:** `big_data.gold.ft_product_performance`
* **Primary Key:** product_id

## Transformations
1. Join and Aggregate Product Metrics - JOIN order_products with products_enriched on product_id, GROUP BY product to calculate times_ordered, times_reordered, estimated_revenue_usd, reorder_rate
2. Create Department Performance - Aggregate by department for total_orders, total_revenue_usd, avg_reorder_rate

## Data Quality
* **Technical:** Product count > 40K, NOT NULL (product_id, times_ordered), Department count match
* **Business:** All products have positive revenue, Department coverage analysis

## Persistence
Writes to Delta tables **only if all validations pass**.

### SETUP

In [0]:
%run ../UTILS/utils

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Schema configuration
silver_schema = "big_data.silver"
gold_schema = "big_data.gold"

# Source tables
source_table_orders = "order_products"
source_table_products = "products_enriched"

# Target tables (fact tables with ft_ prefix)
target_table_products = "ft_product_performance"
target_table_departments = "ft_department_performance"

# Primary Key columns
primary_key_columns = ["product_id"]

# Critical columns (NOT NULL required)
critical_columns = ["product_id", "times_ordered"]

# Validation thresholds
expected_metrics = {
    "min_products": 40_000,
    "expected_departments": 21
}

# Create Gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Configuration:")
print(f"  Source schema: {silver_schema}")
print(f"  Source tables: {source_table_orders}, {source_table_products}")
print(f"  Target: {gold_schema}.{target_table_products}")
print(f"  Primary Key: {', '.join(primary_key_columns)}")

### TRANSFORMATION

In [0]:
print("Step 1: Aggregating product metrics...")

order_products = spark.table(f"{silver_schema}.{source_table_orders}")
products = spark.table(f"{silver_schema}.{source_table_products}")

product_perf = order_products.join(
    products, "product_id", "left"
).groupBy(
    "product_id", "product_name", "department", "aisle", "price_usd"
).agg(
    F.count("*").alias("times_ordered"),
    F.sum(F.when(F.col("reordered") == True, 1).otherwise(0)).alias("times_reordered"),
    F.round(F.sum("price_usd"), 2).alias("estimated_revenue_usd"),
    F.round(F.sum(F.when(F.col("reordered") == True, 1).otherwise(0)) / F.count("*") * 100, 2).alias("reorder_rate")
).withColumn(
    "_gold_timestamp", F.current_timestamp()
)

print(f"  Products aggregated: {product_perf.count():,}")

In [0]:
print("Step 2: Aggregating department metrics...")

# Aggregate by department
dept_perf_gold = product_perf.groupBy("department").agg(
    F.sum("times_ordered").alias("total_orders"),
    F.round(F.sum("estimated_revenue_usd"), 2).alias("total_revenue_usd"),
    F.round(F.avg("reorder_rate"), 2).alias("avg_reorder_rate")
).withColumn(
    "_gold_timestamp", F.current_timestamp()
)

print(f"  Departments aggregated: {dept_perf_gold.count():,}")
print("\nPreview - Department Performance (Top 5 by Revenue):")
dept_perf_gold.orderBy(F.desc("total_revenue_usd")).show(5, truncate=False)

In [0]:
# Create final DataFrame for validation and persistence
df_result = product_perf

print(f"\nFinal DataFrame 'df_result' created: {df_result.count():,} rows")
print("\nReady for validation and persistence")

### DATA QUALITY

In [0]:
print_validation_header("Product Performance - Technical Validations")

# Initialize validation flag
validation_technical = True

# 1. Product count check
product_count = df_result.count()
print(f"\nProduct count: {product_count:,}")
print(f"Expected: >= {expected_metrics['min_products']:,}\n")

if product_count >= expected_metrics["min_products"]:
    status = "PASS"
    msg = f"Product count ({product_count:,}) >= {expected_metrics['min_products']:,}"
else:
    status = "FAIL"
    msg = f"Product count ({product_count:,}) < {expected_metrics['min_products']:,}"
    validation_technical = False
print_check_result("PRODUCT COUNT (>= 40K)", status, msg)

# 2. NOT NULL checks
print("\n2. NOT NULL Validations:")
status, failed, msg = check_not_null(df_result, critical_columns)
print_check_result(f"NOT NULL ({len(critical_columns)} columns)", status, msg, failed)
if status == "FAIL":
    validation_technical = False

# 3. Department count check
print("\n3. Department Count Validation:")
dept_count = dept_perf_gold.count()
print(f"  Department count: {dept_count}")
print(f"  Expected: {expected_metrics['expected_departments']}\n")

if dept_count == expected_metrics["expected_departments"]:
    status = "PASS"
    msg = f"Department count matches expected ({dept_count})"
else:
    status = "FAIL"
    msg = f"Department count mismatch: {dept_count} vs {expected_metrics['expected_departments']}"
    validation_technical = False
print_check_result("DEPARTMENT COUNT", status, msg)

total_rows = product_count

print("\n" + "="*60)
if validation_technical:
    print("SUCCESS: Technical validations PASSED")
else:
    print("FAILURE: Technical validations FAILED")
print("="*60)

In [0]:
print_validation_header("Product Performance - Business Validations")

# Initialize business validation flag
validation_business = True

# 1. All products have positive revenue
print("\n1. Business Rule - Revenue Validation:")
negative_revenue_count = df_result.filter(
    F.col("estimated_revenue_usd") <= 0
).count()

if negative_revenue_count == 0:
    status = "PASS"
    msg = "All products have positive revenue"
else:
    status = "FAIL"
    msg = f"{negative_revenue_count} products with non-positive revenue"
    validation_business = False
print_check_result("REVENUE > 0", status, msg, negative_revenue_count)

# 2. Department coverage
print("\n2. Business Rule - Department Coverage:")
top_depts = dept_perf_gold.orderBy(F.desc("total_revenue_usd")).limit(5).collect()
print("\n  Top 5 Departments by Revenue:")
for dept in top_depts:
    print(f"    - {dept['department']}: ${dept['total_revenue_usd']:,.2f}")

status = "PASS"
msg = f"All {expected_metrics['expected_departments']} departments have data"
print_check_result("DEPARTMENT COVERAGE", status, msg)

print("\n" + "="*60)
if validation_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
# Combine technical and business validation results using UTILS orchestrator
validation_passed = combined_validation_result(validation_technical, validation_business)

### PERSISTENCE

In [0]:
# Conditionally persist to Delta tables using UTILS function
if validation_passed:
    # Persist product performance
    persist_to_delta(df_result, f"{gold_schema}.{target_table_products}")
    
    # Persist department performance
    print("\nPersisting department performance table...")
    dept_perf_gold.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{gold_schema}.{target_table_departments}")
    
    dept_count = spark.table(f"{gold_schema}.{target_table_departments}").count()
    
    print("\n" + "="*60)
    print("SUCCESS: Both tables persisted to Gold layer")
    print("="*60)
    print(f"\nFinal Statistics:")
    print(f"  Product Table: {gold_schema}.{target_table_products} ({total_rows:,} rows)")
    print(f"  Department Table: {gold_schema}.{target_table_departments} ({dept_count} rows)")
    print(f"  Type: Fact tables (product and department metrics)")
    print(f"  Format: Delta")
    print(f"\nNext Step: Query for product analytics and insights")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - tables NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")